<div style="background: linear-gradient(135deg, #0f2027 0%, #1a3a2a 50%, #0d4f3c 100%); padding: 40px 36px; border-radius: 16px; color: white; font-family: 'Segoe UI', sans-serif; border-left: 6px solid #22c55e;">
  <div style="font-size: 12px; color: #86efac; letter-spacing: 3px; text-transform: uppercase; font-weight: 600;">
    SCY1101 — Evaluación Parcial 2 · Notebook 2 de 6
  </div>
  <h1 style="margin: 14px 0 8px; font-size: 32px; font-weight: 700; letter-spacing: -0.5px; color: white;">
    🤖 Modelado Supervisado: 15 Candidatos
  </h1>
  <p style="color: #bbf7d0; margin: 0 0 24px; font-size: 16px; line-height: 1.6;">
    Los datos están certificados. Ahora la pregunta es: ¿qué algoritmo detecta mejor las incidencias<br>
    y cuál estima con menor error los días de tránsito?
  </p>
  <div style="display: flex; gap: 12px; flex-wrap: wrap;">
    <span style="background: rgba(34,197,94,0.2); color: #86efac; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(34,197,94,0.3);">🏁 15 modelos clasificación</span>
    <span style="background: rgba(34,197,94,0.2); color: #86efac; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(34,197,94,0.3);">📏 15 modelos regresión</span>
    <span style="background: rgba(34,197,94,0.2); color: #86efac; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(34,197,94,0.3);">🔁 Validación cruzada</span>
    <span style="background: rgba(34,197,94,0.2); color: #86efac; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(34,197,94,0.3);">📊 F1 + MAE + Baseline</span>
  </div>
  <div style="margin-top: 24px; background: rgba(255,255,255,0.07); border-radius: 10px; padding: 12px 18px; font-size: 13px; color: #86efac;">
    <strong>Progreso del proyecto:</strong>
    <div style="background: rgba(255,255,255,0.15); border-radius: 10px; height: 6px; margin: 8px 0 4px;">
      <div style="background: linear-gradient(90deg, #22c55e, #4ade80); width: 34%; height: 6px; border-radius: 10px;"></div>
    </div>
    <div style="display: flex; justify-content: space-between; font-size: 11px; color: #bbf7d0; opacity: 0.8;">
      <span>✅ EDA</span><span>▶ Modelos</span><span>Evaluación</span><span>Tuning</span><span>Clustering</span><span>Final</span>
    </div>
  </div>
</div>

---

## 🧭 La historia de este notebook

<div style="background: #f0fdf4; border-left: 5px solid #22c55e; padding: 20px 24px; border-radius: 0 12px 12px 0; margin: 16px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #14532d; font-size: 15px;">🏁 El torneo de modelos</strong><br><br>
  <span style="color: #166534; font-size: 14px; line-height: 1.7;">
    Con 866 envíos certificados para clasificación y 790 para regresión, ahora enfrentamos a <strong>15 algoritmos candidatos</strong>
    en cada problema. Todos compiten bajo las mismas condiciones: mismos datos, misma validación cruzada, mismas métricas.<br><br>
    El ganador de clasificación no es el que tiene más <em>accuracy</em> — es el que mejor detecta envíos con incidencia real.
    El ganador de regresión no es el más complejo — es el que se equivoca menos días al estimar el tránsito.
  </span>
</div>

**Dos preguntas de negocio, dos competencias:**

| Problema | Pregunta | Target | Métrica clave |
|----------|----------|--------|---------------|
| Clasificación | ¿Este envío tendrá incidencia? | `tiene_incidencia` | **F1 clase positiva** |
| Regresión | ¿Cuántos días tardará? | `dias_en_transito` | **MAE (en días)** |

> **¿Por qué F1 y no accuracy?** Si el 80% de los envíos no tienen incidencia, un modelo que predice siempre "sin incidencia" tiene 80% de accuracy — pero es inútil. F1 penaliza eso.

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "01_raw"
DATA_MODEL = PROJECT_ROOT / "data" / "05_model_input"
DATA_OUTPUT = PROJECT_ROOT / "data" / "07_model_output"
DATA_REPORT = PROJECT_ROOT / "data" / "08_reporting"
DATA_MODELS = PROJECT_ROOT / "data" / "06_models"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)


def read_csv(path):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


def clean_text_columns(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.replace("Ã³", "o", regex=False)
    return df


---

## 1. 📂 Cargar rankings y datasets ML

<div style="background: #fefce8; border-left: 5px solid #eab308; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Qué cargamos aquí?</strong><br>
  <span style="font-size: 13px; color: #713f12;">
    Los rankings de modelos ya calculados por el pipeline Kedro de <code>model_training</code>,
    junto con los datasets ML certificados para reproducir métricas en este notebook.
    Los resultados no se recalculan desde cero — vienen del pipeline reproducible.
  </span>
</div>

In [ ]:
rank_clf = read_csv(DATA_OUTPUT / "ranking_modelos_clasificacion.csv")
rank_reg = read_csv(DATA_OUTPUT / "ranking_modelos_regresion.csv")
clf_data = read_csv(DATA_MODEL / "model_input_clasificacion.csv")
reg_data = read_csv(DATA_MODEL / "model_input_regresion.csv")
print(f"Clasificacion: {len(rank_clf)} modelos evaluados")
print(f"Regresion: {len(rank_reg)} modelos evaluados")


---

## 2. 🎯 Objetivos supervisados del proyecto

<div style="background: #f0f9ff; border-left: 5px solid #0ea5e9; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Por qué dos problemas separados?</strong><br>
  <span style="font-size: 13px; color: #0c4a6e;">
    Clasificación y regresión responden preguntas distintas con impactos distintos.
    Anticipar una incidencia (clasificación) permite actuar preventivamente.
    Estimar el tiempo de tránsito (regresión) permite planificar operaciones y compromisos con clientes.
    Ambos modelos se entrenan sobre datasets separados para evitar contaminación entre targets.
  </span>
</div>

In [3]:
objetivos = pd.DataFrame([
    {
        "problema": "Clasificacion binaria",
        "pregunta_negocio": "Podemos anticipar si un envio tendra incidencia?",
        "target": "tiene_incidencia",
        "metrica_principal": "f1 clase positiva",
        "justificacion": "Equilibra precision y recall para eventos minoritarios."
    },
    {
        "problema": "Regresion",
        "pregunta_negocio": "Podemos estimar los dias de transito?",
        "target": "dias_en_transito",
        "metrica_principal": "MAE",
        "justificacion": "Se interpreta directamente en dias."
    },
])
objetivos


,problema,pregunta_negocio,target,metrica_principal,justificacion
0,Clasificacion binaria,Podemos anticipar si un envio tendra incidencia?,tiene_incidencia,f1 clase positiva,Equilibra precision y recall para eventos mino...
1,Regresion,Podemos estimar los dias de transito?,dias_en_transito,MAE,Se interpreta directamente en dias.


---

## 3. 🏆 Torneo de clasificación — ¿quién detecta incidencias?

<div style="background: #fdf4ff; border-left: 5px solid #a855f7; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>Regla del torneo:</strong><br>
  <span style="font-size: 13px; color: #581c87;">
    Gana quien maximiza el <strong>F1 de la clase positiva</strong> (envíos con incidencia) en validación cruzada.
    El DummyClassifier actúa como piso mínimo — cualquier modelo útil debe superarlo.
    15 candidatos compiten: desde modelos simples como GaussianNB hasta conjuntos como GradientBoosting.
  </span>
</div>

In [4]:
def nombre_modelo_limpio(nombre):
    return str(nombre).split("_", 1)[-1]


rank_clf_viz = rank_clf.copy()
rank_clf_viz["modelo_claro"] = rank_clf_viz["modelo"].apply(nombre_modelo_limpio)
rank_clf_tabla = rank_clf_viz[[
    "posicion", "modelo_claro", "accuracy", "precision", "recall", "f1", "roc_auc"
]].rename(columns={
    "modelo_claro": "modelo",
    "accuracy": "accuracy_general",
    "precision": "precision_incidencia",
    "recall": "recall_incidencia",
    "f1": "f1_incidencia",
    "roc_auc": "roc_auc"
})
display(rank_clf_tabla)

top_plot = rank_clf_viz.sort_values("f1", ascending=True)
plt.figure(figsize=(11, 7))
ax = sns.barplot(data=top_plot, y="modelo_claro", x="f1", color="#4C78A8")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
plt.title("Capacidad de los modelos para detectar envíos con incidencia")
plt.xlabel("F1 de la clase 'con incidencia' (mayor es mejor)")
plt.ylabel("Modelo candidato")
plt.xlim(0, max(top_plot["f1"].max() * 1.18, 0.35))
plt.tight_layout()
plt.show()


NameError: name 'rank_clf' is not defined

<div style="background: #fdf4ff; border: 1px solid #e9d5ff; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #581c87;">📊 Cómo leer este ranking</strong><br><br>
  <span style="color: #4a1272; font-size: 14px; line-height: 1.7;">
    Cada barra es el F1 de la clase "con incidencia" — más alto significa mejor detección de riesgo real.<br><br>
    <strong>¿Por qué GaussianNB lidera?</strong> Porque tiene el mayor F1 para la clase minoritaria —
    logra detectar más incidencias reales que los demás, aunque su precisión no sea perfecta.<br><br>
    <strong>Para la defensa:</strong> No digas "el mejor modelo es el que tiene más accuracy".
    Aquí importa detectar riesgos operativos — por eso miramos F1 y recall de la clase positiva,
    y el baseline con F1=0 confirma que hay que superar algo concreto, no solo un número abstracto.
  </span>
</div>

---

## 4. 📏 Torneo de regresión — ¿quién estima mejor los días?

<div style="background: #fff7ed; border-left: 5px solid #f59e0b; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>Regla del torneo:</strong><br>
  <span style="font-size: 13px; color: #78350f;">
    Gana quien minimiza el <strong>MAE en días</strong> en validación cruzada.
    El DummyRegressor (predice siempre el promedio de días) es el piso mínimo.
    15 candidatos compiten, incluyendo modelos lineales, de vecinos, de árbol y conjuntos.
  </span>
</div>

In [ ]:
rank_reg_viz = rank_reg.copy()
rank_reg_viz["modelo_claro"] = rank_reg_viz["modelo"].apply(nombre_modelo_limpio)
rank_reg_tabla = rank_reg_viz[[
    "posicion", "modelo_claro", "mae", "rmse", "r2"
]].rename(columns={
    "modelo_claro": "modelo",
    "mae": "MAE_dias",
    "rmse": "RMSE_dias",
    "r2": "R2"
})
display(rank_reg_tabla)

top_plot = rank_reg_viz.sort_values("mae", ascending=False)
plt.figure(figsize=(11, 7))
ax = sns.barplot(data=top_plot, y="modelo_claro", x="mae", color="#F58518")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
plt.title("Error promedio al estimar días de tránsito")
plt.xlabel("MAE en días (menor es mejor)")
plt.ylabel("Modelo candidato")
plt.tight_layout()
plt.show()


<div style="background: #fff7ed; border: 1px solid #fed7aa; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #9a3412;">📊 Cómo leer este ranking</strong><br><br>
  <span style="color: #7c2d12; font-size: 14px; line-height: 1.7;">
    El MAE se interpreta directamente en días. Si un modelo tiene <strong>MAE = 1.44</strong>,
    significa que en promedio se equivoca 1.44 días al estimar el tiempo de tránsito.<br><br>
    <strong>Ejemplo concreto:</strong> si un envío tarda realmente 7 días, el modelo podría predecir entre 5.6 y 8.4 días.
    No es precisión quirúrgica, pero permite una planificación inicial útil.<br><br>
    <strong>KNeighborsRegressor lidera</strong> con MAE = 1.44 días — apenas por debajo del DummyRegressor (MAE ≈ 1.48 días).
    El margen es pequeño, lo que se reporta con honestidad como limitación de las variables disponibles.
  </span>
</div>

---

## 5. ⚖️ Comparación contra el baseline

<div style="background: #f0f9ff; border-left: 5px solid #0284c7; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Por qué comparar contra un modelo tonto?</strong><br>
  <span style="font-size: 13px; color: #0c4a6e;">
    El baseline es la respuesta más simple posible: "predice siempre la clase mayoritaria" (clasificación)
    o "predice siempre el promedio de días" (regresión). Si un modelo sofisticado no supera eso,
    no está aprendiendo nada útil — solo memoriza la distribución de los datos.
    Esta comparación es obligatoria para una defensa técnica seria.
  </span>
</div>

In [ ]:
baseline_clf = rank_clf[rank_clf["modelo"].str.contains("Dummy", na=False)].iloc[0]
top_clf = rank_clf.iloc[0]
baseline_reg = rank_reg[rank_reg["modelo"].str.contains("Dummy", na=False)].iloc[0]
top_reg = rank_reg.iloc[0]

comparacion_baseline = pd.DataFrame([
    {
        "problema": "Detectar incidencias",
        "modelo": "Baseline: predice sin incidencia",
        "metrica": "F1 incidencia",
        "valor": baseline_clf["f1"],
        "lectura": "Referencia minima; no detecta incidencias."
    },
    {
        "problema": "Detectar incidencias",
        "modelo": f"Mejor candidato: {nombre_modelo_limpio(top_clf['modelo'])}",
        "metrica": "F1 incidencia",
        "valor": top_clf["f1"],
        "lectura": "Detecta mejor la clase de riesgo que el baseline."
    },
    {
        "problema": "Estimar días de tránsito",
        "modelo": "Baseline: predice promedio",
        "metrica": "MAE días",
        "valor": baseline_reg["mae"],
        "lectura": "Error de referencia usando una prediccion simple."
    },
    {
        "problema": "Estimar días de tránsito",
        "modelo": f"Mejor candidato: {nombre_modelo_limpio(top_reg['modelo'])}",
        "metrica": "MAE días",
        "valor": top_reg["mae"],
        "lectura": "Error promedio del mejor modelo candidato."
    },
])
display(comparacion_baseline)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

clf_base = comparacion_baseline[comparacion_baseline["problema"] == "Detectar incidencias"]
ax0 = sns.barplot(data=clf_base, x="modelo", y="valor", ax=axes[0], color="#4C78A8")
for container in ax0.containers:
    ax0.bar_label(container, fmt="%.3f", padding=3)
axes[0].set_title("¿El modelo detecta incidencias mejor que una regla trivial?")
axes[0].set_xlabel("")
axes[0].set_ylabel("F1 de envíos con incidencia (mayor es mejor)")
axes[0].tick_params(axis="x", rotation=15)

reg_base = comparacion_baseline[comparacion_baseline["problema"] == "Estimar días de tránsito"]
ax1 = sns.barplot(data=reg_base, x="modelo", y="valor", ax=axes[1], color="#F58518")
for container in ax1.containers:
    ax1.bar_label(container, fmt="%.3f", padding=3)
axes[1].set_title("¿El modelo estima días mejor que predecir el promedio?")
axes[1].set_xlabel("")
axes[1].set_ylabel("MAE en días (menor es mejor)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


<div style="background: #f0f9ff; border: 1px solid #bae6fd; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #0c4a6e;">💬 Argumento de defensa listo</strong><br><br>
  <span style="color: #1e3a5f; font-size: 14px; line-height: 1.7;">
    "No basta con decir que entrenamos modelos. Verificamos que los modelos útiles superan una referencia mínima.
    En clasificación, el baseline tiene F1=0 porque no detecta incidencias — el mejor candidato (GaussianNB) sí las detecta.
    En regresión, el baseline predice el promedio de días — KNeighbors reduce ese error en ~0.04 días.
    Cuando la mejora es pequeña, también lo decimos: eso muestra criterio técnico, no debilidad del análisis."
  </span>
</div>

---

## 🏁 Conclusión: los finalistas están seleccionados

<div style="background: linear-gradient(135deg, #0f2027 0%, #1a3a2a 50%, #0d4f3c 100%); padding: 28px 32px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0;">
  <h3 style="margin: 0 0 16px; color: #86efac;">📋 Resumen del torneo</h3>
  <div style="display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 20px;">
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #4ade80;">15 + 15</div>
      <div style="font-size: 12px; color: #bbf7d0; margin-top: 4px;">Modelos candidatos evaluados</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #86efac;">GaussianNB</div>
      <div style="font-size: 12px; color: #bbf7d0; margin-top: 4px;">Finalista clasificación · F1=0.276</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #86efac;">KNeighbors</div>
      <div style="font-size: 12px; color: #bbf7d0; margin-top: 4px;">Finalista regresión · MAE=1.44 días</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #fbbf24;">✅</div>
      <div style="font-size: 12px; color: #bbf7d0; margin-top: 4px;">Baseline superado en ambos problemas</div>
    </div>
  </div>
  <p style="color: #bbf7d0; margin: 0; font-size: 14px; line-height: 1.7;">
    Se cumplió el requisito de comparación amplia: 15 candidatos por problema, con validación cruzada,
    métricas múltiples y comparación contra baseline. Los finalistas avanzan a la fase de optimización.
  </p>
</div>

<div style="background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 10px; padding: 16px 20px; margin-top: 16px; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #1e40af;">➡️ Siguiente: Notebook 03 — Evaluación de Modelos Finales</strong><br>
  <span style="color: #1e3a8a; font-size: 14px;">
    GaussianNB y KNeighbors pasan a evaluación profunda: matriz de confusión, curva ROC,
    análisis de errores reales y traducción de métricas a decisiones de negocio.
  </span>
</div>